# Week 1: Data Quality Analysis

This notebook performs comprehensive data quality checks on the Telco Customer Churn dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Loaded: {df.shape}")

## 1. Duplicate Check

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Check for duplicate customer IDs
duplicate_ids = df['customerID'].duplicated().sum()
print(f"Duplicate customer IDs: {duplicate_ids}")

if duplicates == 0:
    print("\n✓ No duplicate rows found!")

## 2. Column Type Analysis

In [ ]:
# Separate column types
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("="*50)
print(f"CATEGORICAL COLUMNS ({len(categorical_cols)})")
print("="*50)
for col in categorical_cols:
    print(f"  - {col}")

print("\n" + "="*50)
print(f"NUMERIC COLUMNS ({len(numeric_cols)})")
print("="*50)
for col in numeric_cols:
    print(f"  - {col}")

## 3. Categorical Variables Profile

In [ ]:
# Analyze key categorical variables
key_categorical = ['gender', 'Partner', 'Dependents', 'PhoneService', 'InternetService', 'Contract', 'Churn']

for col in key_categorical:
    print("\n" + "="*50)
    print(f"{col.upper()}")
    print("="*50)
    print(df[col].value_counts())
    print(f"\nUnique values: {df[col].nunique()}")

## 4. Outlier Detection

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(df, column):
    """Detect outliers using Interquartile Range method."""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Check outliers for numeric columns
numeric_features = ['tenure', 'MonthlyCharges']  # TotalCharges needs cleaning first

for col in numeric_features:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print("\n" + "="*50)
    print(f"{col.upper()} OUTLIERS")
    print("="*50)
    print(f"Lower bound: {lower:.2f}")
    print(f"Upper bound: {upper:.2f}")
    print(f"Outliers found: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")

## 5. TotalCharges Issue

In [ ]:
# TotalCharges has empty strings that need to be handled
print("="*50)
print("TOTALCHARGES ANALYSIS")
print("="*50)

# Check for non-numeric values
non_numeric = df[pd.to_numeric(df['TotalCharges'], errors='coerce').isnull() & (df['TotalCharges'] != ' ')]
empty_strings = (df['TotalCharges'] == ' ').sum()

print(f"Empty strings: {empty_strings}")
print(f"Other non-numeric: {len(non_numeric)}")

# Check if empty TotalCharges correlates with tenure=0
empty_mask = df['TotalCharges'] == ' '
print(f"\nEmpty TotalCharges with tenure=0: {(df[empty_mask]['tenure'] == 0).sum()}/{empty_strings}")

## 6. Churn Distribution

In [ ]:
# Analyze churn distribution
print("="*50)
print("CHURN DISTRIBUTION")
print("="*50)

churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage': churn_pct.round(2)
})

print(churn_summary)

print(f"\nChurn Rate: {churn_pct['Yes']:.1f}%")

In [ ]:
# Visualize churn distribution
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Churn', palette='Set2')

# Add value labels on bars
for i, v in enumerate(churn_counts):
    ax.text(i, v + 50, str(v), ha='center', va='bottom', fontsize=12)

plt.title('Customer Churn Distribution', fontsize=14)
plt.xlabel('Churn')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../outputs/churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: outputs/churn_distribution.png")

## 7. Data Quality Summary

In [ ]:
# Save data quality report
with open('../outputs/data_quality_report.txt', 'w') as f:
    f.write("="*50 + "\n")
    f.write("DATA QUALITY REPORT\n")
    f.write("="*50 + "\n\n")
    
    f.write("1. DUPLICATES\n")
    f.write(f"   - Duplicate rows: {duplicates}\n")
    f.write(f"   - Duplicate IDs: {duplicate_ids}\n\n")
    
    f.write("2. MISSING VALUES\n")
    f.write(f"   - Empty TotalCharges: {empty_strings}\n")
    f.write(f"   - All correspond to tenure=0 (new customers)\n\n")
    
    f.write("3. CATEGORICAL VARIABLES\n")
    f.write(f"   - Total categorical: {len(categorical_cols)}\n")
    f.write(f"   - Binary features: {len([c for c in categorical_cols if df[c].nunique() == 2])}\n\n")
    
    f.write("4. NUMERIC VARIABLES\n")
    f.write(f"   - Total numeric: {len(numeric_cols)}\n")
    f.write(f"   - Tenure range: {df['tenure'].min()}-{df['tenure'].max()} months\n")
    f.write(f"   - MonthlyCharges range: ${df['MonthlyCharges'].min():.2f}-${df['MonthlyCharges'].max():.2f}\n\n")
    
    f.write("5. TARGET VARIABLE (CHURN)\n")
    f.write(f"   - Churn rate: {churn_pct['Yes']:.1f}%\n")
    f.write(f"   - Class distribution: Moderately imbalanced\n")

print("✓ Data quality report saved to outputs/data_quality_report.txt")